## Portkey Implementation

In [1]:
import os
import time
import uuid
import json
import time
from dotenv import load_dotenv

print("Import for os time uuid json and dotenv is done")

Import for os time uuid json and dotenv is done


In [2]:
# Helper functions 
def pretty_print(title):
    print("="*100)
    print(f"\n{title}\n")
    print("="*100)
    
def showResult(question,answer,time, label):
    print("="*100 + "\n")
    print(f"Question: {question}")
    print(f"Answer: {answer}")
    print(f"Time: {time}")
    print(f"Label: {label}")
    print("\n" +"="*100 + "\n")
    
    

In [3]:
# Load secrets from .env — never hardcode API keys in this notebook.
# Re-run this cell after editing .env (no kernel restart needed).
from portkey_ai import Portkey, createHeaders, PORTKEY_GATEWAY_URL

load_dotenv(override=True)

REQUIRED_ENV_VARS = (
    "PORTKEY_API_KEY",
    "PORTKEY_PROVIDER",
    "DEFAULT_LLM_MODEL",
    "PORTKEY_RETRY_CONFIG_ID",
)

missing = [name for name in REQUIRED_ENV_VARS if not os.getenv(name)]
if missing:
    raise ValueError(
        "Missing required environment variables: "
        + ", ".join(missing)
        + ". Copy .env.example to .env and set your values."
    )

PORTKEY_API_KEY = os.getenv("PORTKEY_API_KEY")
PORTKEY_PROVIDER = os.getenv("PORTKEY_PROVIDER")
DEFAULT_LLM_MODEL = os.getenv("DEFAULT_LLM_MODEL")
PORTKEY_RETRY_CONFIG_ID = os.getenv("PORTKEY_RETRY_CONFIG_ID")
print("Environment loaded. Model:", DEFAULT_LLM_MODEL)


Environment loaded. Model: gpt-4o-mini


### Basic LLM Calling without Any LLM Gateway

In [4]:

from langchain_openai import ChatOpenAI
question = "What is the capital of France?"
chat_llm = ChatOpenAI(model=DEFAULT_LLM_MODEL, temperature=0)
start_time = time.time()
result = chat_llm.invoke(question)
end_time = time.time()
response_time = end_time - start_time

showResult(question,result.content,response_time,"Direct LLM Calling")



Question: What is the capital of France?
Answer: The capital of France is Paris.
Time: 2.272876501083374
Label: Direct LLM Calling




### LLM Calling with Portkey LLM Gateway
Helper Function and initialization

In [5]:
# Portkey client: API key + provider slug from .env only.
portkey = Portkey(
    api_key=PORTKEY_API_KEY,
    provider=PORTKEY_PROVIDER,
)

from functools import wraps


class TimeLogger:
    """Decorator that returns (result, elapsed_seconds)."""

    def __call__(self, func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            start_time = time.time()
            result = func(*args, **kwargs)
            end_time = time.time()
            return result, end_time - start_time

        return wrapper


**Single Message Calls**

In [6]:

    
@TimeLogger()
def usePortkey(question):
    result = portkey.chat.completions.create(
    model=DEFAULT_LLM_MODEL,     # "gpt-4o-mini"
    messages=[{"role": "user", "content": question}],
    )
    return result

result, response_time = usePortkey(question)
showResult(question,result.choices[0].message.content,response_time,"Portkey LLM Calling")


Question: What is the capital of France?
Answer: The capital of France is Paris.
Time: 1.4978888034820557
Label: Portkey LLM Calling




**Sending MetaData for the better Analysis** (with options)

In [7]:
@TimeLogger()
def usePortkeyWithMetadata(question, metadata):
    result = portkey.with_options(
        metadata=metadata
    ).chat.completions.create(
        model=DEFAULT_LLM_MODEL,
        messages=[
            {
                "role": "user",
                "content": question
            }
        ],
    )
    return result
question = "how to achive consistency and disciple in life"
result, response_time = usePortkeyWithMetadata(question,{
            "user_id": "user-123",
        "feature": "chat",
        "request_id": "req-456"})   

showResult(question,result.choices[0].message.content,response_time,"Portkey LLM Calling with metadata")


Question: how to achive consistency and disciple in life
Answer: Achieving consistency and discipline in life requires a combination of mindset, habits, and strategies. Here are some effective steps to help you cultivate these qualities:

### 1. **Set Clear Goals**
   - **Define Your Objectives**: Establish both short-term and long-term goals that are specific, measurable, achievable, relevant, and time-bound (SMART).
   - **Break Down Goals**: Divide larger goals into smaller, manageable tasks to avoid feeling overwhelmed.

### 2. **Create a Routine**
   - **Establish a Daily Schedule**: Design a routine that includes dedicated time for work, exercise, relaxation, and other activities.
   - **Prioritize Tasks**: Start your day with the most important tasks. Use techniques like the Eisenhower Matrix to prioritize effectively.

### 3. **Develop Healthy Habits**
   - **Consistency Over Perfection**: Focus on making progress rather than being perfect. Develop habits that you can stick to

### Retry Mechanism
This workspace blocks inline JSON configs (`block_inline_config`), so use either:
1. **Application-level retry** (Python wrapper below) — works immediately
2. **Dashboard config** — create a retry config in Portkey, then set `PORTKEY_RETRY_CONFIG_ID=pc-...` in `.env`


**Local Retry Mechanism**

Application-level retry wrapper for transient errors (429/5xx). Loads optional `PORTKEY_RETRY_CONFIG_ID` from `.env` when set.


In [8]:
from portkey_ai.api_resources.exceptions import (
    APIConnectionError,
    InternalServerError,
    RateLimitError,
)

# This workspace blocks inline JSON configs (block_inline_config).
# Use application-level retries below, or set PORTKEY_RETRY_CONFIG_ID in .env
# after creating a retry config in the Portkey dashboard.
MAX_RETRIES = 3
RETRYABLE_STATUS_CODES = {429, 500, 502, 503, 504}


def call_with_retry(fn):
    """Retry transient Portkey/provider failures with exponential backoff."""
    for attempt in range(MAX_RETRIES + 1):
        try:
            return fn()
        except (RateLimitError, InternalServerError, APIConnectionError) as exc:
            status_code = getattr(exc, "status_code", None)
            if status_code not in RETRYABLE_STATUS_CODES and not isinstance(
                exc, APIConnectionError
            ):
                raise
            if attempt == MAX_RETRIES:
                raise
            wait_seconds = 2 ** attempt
            print(
                f"Retry {attempt + 1}/{MAX_RETRIES} in {wait_seconds}s "
                f"(status={status_code})"
            )
            time.sleep(wait_seconds)


@TimeLogger()
def usePortkeyWithMetadataAndRetry(question, metadata):
    def _create_completion():
        client = portkey.with_options(metadata=metadata)
        if PORTKEY_RETRY_CONFIG_ID:
            client = client.with_options(config=PORTKEY_RETRY_CONFIG_ID)

        return client.chat.completions.create(
            model=DEFAULT_LLM_MODEL,
            messages=[{"role": "user", "content": question}],
        )

    return call_with_retry(_create_completion)


question = "how to achieve consistency and discipline in life"
result, response_time = usePortkeyWithMetadataAndRetry(
    question,
    {
        "user_id": "user-123",
        "feature": "chat",
        "request_id": "req-456",
    },
)

showResult(
    question,
    result.choices[0].message.content,
    response_time,
    "Portkey with metadata + retry",
)



Question: how to achieve consistency and discipline in life
Answer: Achieving consistency and discipline in life is a process that requires commitment, self-awareness, and practical strategies. Here are several steps you can take to cultivate these traits:

### 1. **Set Clear Goals**
   - **Define Your Purpose:** Understand why you want to achieve consistency and discipline. This could be related to personal development, health, career, or relationships.
   - **SMART Goals:** Make your goals Specific, Measurable, Achievable, Relevant, and Time-bound. This clarity will help you stay focused.

### 2. **Create a Routine**
   - **Daily Schedule:** Establish a daily routine that outlines when to wake up, work, exercise, and relax. Consistent schedules help form habits.
   - **Prioritize Tasks:** Use tools like to-do lists or planners to prioritize tasks and allocate specific times for each.

### 3. **Start Small**
   - **One Step at a Time:** Begin with small, manageable changes to build m

**Portkey Based Config Implementation**

In [10]:
question = "What is Special Today"

@TimeLogger()
def usePortkeyWithMetadataAndRetry(question, metadata):
    if not PORTKEY_RETRY_CONFIG_ID:
        raise ValueError(
            "Set PORTKEY_RETRY_CONFIG_ID in .env (dashboard config pc-...). "
            "Inline configs are blocked in this workspace."
        )

    client = portkey.with_options(
        metadata=metadata,
        config=PORTKEY_RETRY_CONFIG_ID,
    )

    return client.chat.completions.create(
        model=DEFAULT_LLM_MODEL,
        messages=[{"role": "user", "content": question}],
    )


result, response_time = usePortkeyWithMetadataAndRetry(
    question,
    {
        "user_id": "user-123",
        "feature": "chat",
        "request_id": "req-456",
    },
)

showResult(
    question,
    result.choices[0].message.content,
    response_time,
    "Portkey with metadata + retry(Portkey Config)",
)


Question: What is Special Today
Answer: Could you please provide more context on what you mean by "Special Today"? Are you asking about a specific event, promotion, holiday, or something else?
Time: 3.3167290687561035
Label: Portkey with metadata + retry(Portkey Config)




## Implmentation of TimeOut

In [ ]:
Code for the Time out will Remains Same 
Only change will be inside the Config file : 


## Implementation of FallBack

Code for the Time out will Remains Same 
Only change will be inside the Config file : 


## Streaming Portkey Implementation


## LLM Portkey Implementation


### Check List

- [x] Implement Simple Chat Conversation using the Portkey

- [x] Implement Caching

- [x] Implement Loadbalancing

- [x] Need to understand How Tracing is done

- [x] Need to know how token tracing can be Implemented  

- [x] Routing using the Portkey  

- [x] Retries Mechanism  

- [x] Streamin Response  